# Descriptive Statistics

**DS4DH Practice Pack · Module 02 — Describing and Comparing Data**

*Technique:* Mean, median, standard deviation, IQR — and when each misleads

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/02a_descriptive_stats.ipynb)

Data: `merged_dataset.csv`, `city_summary.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv', 'city_summary.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
df_city  = pd.read_csv('city_summary.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Four summary numbers, and the question of which to report.

Mean and median answer different questions. Standard deviation and IQR answer
different questions. In a skewed distribution — which housing cost data always
is — choosing the wrong one of each pair produces a number that is arithmetically
correct and substantively misleading.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
d = base.dropna(subset=['Renter'])

print(f'{"City":<12}{"n":>5}{"mean":>8}{"median":>8}{"mean-med":>10}{"std":>8}{"IQR":>8}')
print('-' * 59)
for city in CITIES:
    s = d[d['cma'] == city]['Renter']
    iqr = s.quantile(0.75) - s.quantile(0.25)
    print(f'{city:<12}{len(s):>5}{s.mean():>8.1f}{s.median():>8.1f}'
          f'{s.mean() - s.median():>+10.2f}{s.std():>8.2f}{iqr:>8.2f}')

## Reading the mean−median column

`mean − median` is a one-number skew detector.

- **positive** → a right tail: a few CSDs with very high renter burden pull the mean up
- **near zero** → roughly symmetric
- **negative** → a left tail

Where it is positive, the mean overstates what a typical municipality looks like.

In [ ]:
# Which places create the right tail?
top = d.nlargest(6, 'Renter')[['geography_name', 'cma', 'Renter', 'tot_pop']]
print('Highest renter STIR:')
print(top.to_string(index=False))
print()
print('Now drop them and watch the mean move, while the median barely does.')
trimmed = d.drop(top.index)
print(f'  mean   with: {d["Renter"].mean():.2f}   without: {trimmed["Renter"].mean():.2f}')
print(f'  median with: {d["Renter"].median():.2f}   without: {trimmed["Renter"].median():.2f}')

### 🔧 Your turn 1

Change `nlargest(6, ...)` to `nlargest(15, ...)` and re-run.

How far does the median move compared to the mean? That difference in
sensitivity *is* the reason to prefer the median for a typical-case claim.

## Standard deviation vs IQR

The same split applies to spread. Standard deviation squares every deviation, so
one extreme CSD contributes disproportionately. The IQR ignores the tails
entirely and describes the middle half.

In [ ]:
for city in CITIES:
    s = d[d['cma'] == city]['Renter']
    iqr = s.quantile(0.75) - s.quantile(0.25)
    # For a normal distribution IQR ≈ 1.35·σ. A ratio far from that is a
    # direct signal that the tails, not the middle, are driving σ.
    print(f'{city:<12} std={s.std():>5.2f}  IQR={iqr:>5.2f}  IQR/std={iqr / s.std():>5.2f}'
          f'   (≈1.35 if normal)')

## Cross-checking the pre-computed file

`city_summary.csv` ships with averages already computed. Never trust a summary
file you did not build without checking it against the source.

In [ ]:
tot = df_city[df_city['immigrant_status'] == 'Total Immigrant Status']
print(f'{"City":<12}{"summary file":>14}{"recomputed":>13}{"diff":>9}')
print('-' * 48)
for city in CITIES:
    theirs = tot[tot['cma'] == city]['avg_renter_stir']
    mine = d[d['cma'] == city]['Renter'].mean()
    t = float(theirs.iloc[0]) if len(theirs) else float('nan')
    print(f'{city:<12}{t:>14.2f}{mine:>13.2f}{t - mine:>+9.2f}')
print()
print('Differences here are not errors — they come from a different choice about')
print('which rows to average. Notebook 09b is entirely about that choice.')

### 🔧 Your turn 2

Report a single figure for "renter housing burden in Montréal" to a housing
agency. Which number do you send, and what one sentence goes with it?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** The mean keeps moving as you remove more high values; the median
moves very little. That is the definition of robustness — the median depends on
the rank of the middle observation, not the value of the extremes. If your claim
is "a typical municipality looks like X", the median is the number that supports
it.

**Your turn 2.** Send the median, with the IQR and the n. Something like:

> Across the 88 Montréal CSDs with reported renter data, the median renter
> spends about 26% of household income on shelter (middle half: 23–29%).

The mean invites a follow-up question you cannot answer well ("is that pulled up
by a few places?"), whereas the median plus IQR pre-empts it. Reporting the mean
alone, from a right-skewed distribution, describes a municipality that may not
exist.

</details>

## Where this stops

You can describe one group. You cannot yet say whether two groups differ — the
next notebook adds the shape of the distribution, and 02c adds the comparison.